# Iteration 1 – BiGRU neural network

| | Iteration 1 |
|---|---|
| **Model** | bidirectional GRU neural network, trained from scratch |
| **Libraries** | `torch` (PyTorch), `numpy`, `pandas` |
| **How text becomes numbers** | words → word vectors learned from scratch → the GRU reads the words in order (max 256 words) |
| **Input** | description + rank |
| **Output** | 3,705 keywords + 100 traits and traditions |
| **Data** | Foundry export, 1,920 spells, 80/10/10 split |
| **Result** | micro-F1 **0.388** (always guessing `concentrate, manipulate` gives 0.473) |
| **Test: only rank + description** | micro-F1 **0.388** (rank + description is already its input) |
| **Status** | ❌ worse than a constant guess |

(AI geenrated tabs)

## PF2e Spell Metadata - standalone multi-label model

Baseline version without a second model. One neural network receives the spell description and rank, then predicts the complete label set: official `traits` and keywords. Labels share one output space and are separated during decoding by the `trait:` and `keyword:` prefixes.

The model returns up to five traits and five keywords by default. This standalone version is a baseline for comparison with a two-stage architecture.

In [1]:
# This notebook intentionally has no second-model calls; the full prediction
# path is implemented here.
from __future__ import annotations

import json
import random
import re
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset

ROOT = Path.cwd()
DATA_DIR = ROOT.parents[1] / "data" / "spellsdata"
ARTIFACT_DIR = ROOT / "artifacts"
SEED = 42
MAX_TEXT_LENGTH = 256
MAX_VOCAB_SIZE = 12000
MIN_KEYWORD_DOCUMENT_FREQUENCY = 3
DEFAULT_TOP_K_KEYWORDS = 5
DEFAULT_TOP_K_TRAITS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"device={DEVICE}, data_dir={DATA_DIR}")
print(f"config: max_tokens={MAX_TEXT_LENGTH}, max_vocab={MAX_VOCAB_SIZE}, min_keyword_df={MIN_KEYWORD_DOCUMENT_FREQUENCY}")
print(f"outputs: top_k_keywords={DEFAULT_TOP_K_KEYWORDS}, top_k_traits={DEFAULT_TOP_K_TRAITS}")

device=cpu, data_dir=/Users/bohdanlaskavyi/Fontys/year2september3/Personall challenge/PF2e-splells-creations/data/spellsdata
config: max_tokens=256, max_vocab=12000, min_keyword_df=3
outputs: top_k_keywords=5, top_k_traits=5


## Stage 2: dataset and shared label space

Each Foundry JSON provides the description, rank, traits, and traditions. Keywords are extracted from the cleaned text. This version has one output head: all labels are stored in one list, while prefixes preserve the label type for JSON decoding.

In [2]:
TOKEN_PATTERN = re.compile(r"[a-z][a-z0-9'-]{2,}")
HTML_PATTERN = re.compile(r"<[^>]+>")
UUID_PATTERN = re.compile(r"@UUID\[[^]]+\](?:\{([^}]+)\})?")
STOP_WORDS = {
    "the", "and", "for", "that", "this", "with", "from", "into", "your", "you",
    "are", "can", "each", "when", "then", "than", "has", "have", "will", "its",
    "not", "one", "only", "once", "all", "any", "use", "used", "using", "spell",
}

def clean_text(value: str) -> str:
    value = HTML_PATTERN.sub(" ", value or "")
    value = UUID_PATTERN.sub(lambda match: match.group(1) or " ", value)
    return " ".join(value.lower().split())

def tokenize(value: str) -> list[str]:
    return [token for token in TOKEN_PATTERN.findall(clean_text(value)) if token not in STOP_WORDS]

def load_spells(data_dir: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for path in sorted(data_dir.rglob("*.json")):
        if path.name == "_folders.json":
            continue
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
            system = payload.get("system", {})
            text = clean_text(system.get("description", {}).get("value", ""))
            traits_data = system.get("traits", {})
            official_labels = [*traits_data.get("value", []), *traits_data.get("traditions", [])]
            traits = sorted({str(label).casefold() for label in official_labels if label})
            if text and traits:
                rows.append({
                    "name": payload.get("name", path.stem),
                    "text": text,
                    "tokens": tokenize(text),
                    "rank": int(system.get("level", {}).get("value", 0) or 0),
                    "traits": traits,
                })
        except (OSError, json.JSONDecodeError, TypeError, ValueError) as error:
            print(f"Skipped {path}: {error}")
    return pd.DataFrame(rows)

print("\n[1/5] Loading and cleaning JSON spell files...")
df = load_spells(DATA_DIR)
print(f"  loaded descriptions with labels: {len(df):,}")
document_frequency = Counter(token for tokens in df["tokens"] for token in set(tokens))
keyword_labels = sorted(token for token, count in document_frequency.items()
                        if count >= MIN_KEYWORD_DOCUMENT_FREQUENCY)
keyword_set = set(keyword_labels)
df["keywords"] = df["tokens"].apply(lambda tokens: sorted(set(tokens) & keyword_set))
df = df[df["keywords"].map(bool)].reset_index(drop=True)
trait_labels = sorted({trait for labels in df["traits"] for trait in labels})
keyword_to_id = {label: index for index, label in enumerate(keyword_labels)}
trait_to_id = {label: index for index, label in enumerate(trait_labels)}

print(f"usable spells={len(df)}")
print(f"separate label spaces: {len(keyword_labels)} keywords + {len(trait_labels)} traits")
print(f"  keyword rule: token appears in at least {MIN_KEYWORD_DOCUMENT_FREQUENCY} documents")
print(f"  example tokens: {df.iloc[0]["tokens"][:10]}")
print(df[["name", "rank", "keywords", "traits"]].head(3).to_string(index=False))


[1/5] Loading and cleaning JSON spell files...


  loaded descriptions with labels: 1,920
usable spells=1920
separate label spaces: 3705 keywords + 100 traits
  keyword rule: token appears in at least 3 documents
  example tokens: ['utter', 'phrases', 'unknown', 'tongue', 'assaulting', 'minds', 'those', 'nearby', 'target', 'must']
                     name  rank                                                                                                                                                                                                                                                                          keywords                                           traits
        Aberrant Whispers     3 [action, additional, attempt, casting, critical, emanation's, failure, feet, heightened, immune, increase, increases, initial, minds, minute, must, nearby, radius, regardless, result, save, stupefied, success, target, temporarily, those, tongue, unaffected, unknown, utter] [auditory, concentrate, focus, mental, sorcerer]
Accele

## Stage 3: vocabulary, split, and DataLoader

Targets use two multi-hot vectors: one for keywords and one for traits. The vocabulary is built only from the training split. Rank is passed to the model as a separate numeric feature.

In [3]:
def split_frame(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    shuffled = frame.sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_start = int(len(shuffled) * 0.8)
    valid_start = int(len(shuffled) * 0.9)
    return shuffled.iloc[:test_start], shuffled.iloc[test_start:valid_start], shuffled.iloc[valid_start:]

train_df, valid_df, test_df = split_frame(df)
train_token_frequency = Counter(token for tokens in train_df["tokens"] for token in tokens)
vocabulary_tokens = [token for token, _ in train_token_frequency.most_common(MAX_VOCAB_SIZE - 2)]
token_to_id = {"<PAD>": 0, "<UNK>": 1, **{token: index + 2 for index, token in enumerate(vocabulary_tokens)}}

def multi_hot(labels: list[str], mapping: dict[str, int]) -> Tensor:
    vector = torch.zeros(len(mapping), dtype=torch.float32)
    for label in labels:
        vector[mapping[label]] = 1.0
    return vector

class SpellDataset(Dataset[dict[str, Tensor]]):
    def __init__(self, frame: pd.DataFrame) -> None:
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict[str, Tensor]:
        row = self.frame.iloc[index]
        ids = [token_to_id.get(token, token_to_id["<UNK>"]) for token in row["tokens"][:MAX_TEXT_LENGTH]]
        ids += [token_to_id["<PAD>"]] * (MAX_TEXT_LENGTH - len(ids))
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "rank": torch.tensor([row["rank"] / 10.0], dtype=torch.float32),
            "keyword_labels": multi_hot(row["keywords"], keyword_to_id),
            "trait_labels": multi_hot(row["traits"], trait_to_id),
        }

train_loader = DataLoader(SpellDataset(train_df), batch_size=32, shuffle=True)
valid_loader = DataLoader(SpellDataset(valid_df), batch_size=64)
test_loader = DataLoader(SpellDataset(test_df), batch_size=64)

keyword_counts = torch.tensor([sum(label in row["keywords"] for _, row in train_df.iterrows()) for label in keyword_labels], dtype=torch.float32)
trait_counts = torch.tensor([sum(label in row["traits"] for _, row in train_df.iterrows()) for label in trait_labels], dtype=torch.float32)
keyword_pos_weight = ((len(train_df) - keyword_counts) / keyword_counts.clamp_min(1)).clamp(max=20.0).to(DEVICE)
trait_pos_weight = ((len(train_df) - trait_counts) / trait_counts.clamp_min(1)).clamp(max=20.0).to(DEVICE)
print("\n[2/5] Building vocabulary and train/validation/test loaders...")
print(f"  split sizes: train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}")
print(f"  vocabulary={len(token_to_id)}, keyword width={len(keyword_labels)}, trait width={len(trait_labels)}")
print(f"  batch sizes: train={train_loader.batch_size}, valid={valid_loader.batch_size}, test={test_loader.batch_size}")
print(f"  tensor example: input_ids=({MAX_TEXT_LENGTH},), rank=(1,), targets=({len(keyword_labels)} + {len(trait_labels)})")


[2/5] Building vocabulary and train/validation/test loaders...
  split sizes: train=1536, valid=192, test=192
  vocabulary=8702, keyword width=3705, trait width=100
  batch sizes: train=32, valid=64, test=64
  tensor example: input_ids=(256,), rank=(1,), targets=(3705 + 100)


## Stage 4: shared encoder with two output heads

The shared encoder processes the text and rank. Two output heads then predict keywords and traits separately. Their predictions are exported together for the next model.

In [4]:
class SpellTagger(nn.Module):
    def __init__(self, vocab_size: int, keyword_count: int, trait_count: int, embedding_dim: int = 96, hidden_dim: int = 96) -> None:
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.encoder = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.shared_classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 128),
        )
        self.keyword_head = nn.Linear(128, keyword_count)
        self.trait_head = nn.Linear(128, trait_count)

    def forward(self, input_ids: Tensor, rank: Tensor) -> Tensor:
        encoded, _ = self.encoder(self.embedding(input_ids))
        mask = input_ids.ne(0).unsqueeze(-1)
        pooled = (encoded * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        shared = self.shared_classifier(torch.cat([pooled, rank], dim=1))
        return {"keywords": self.keyword_head(shared), "traits": self.trait_head(shared)}

model = SpellTagger(len(token_to_id), len(keyword_labels), len(trait_labels)).to(DEVICE)
keyword_loss = nn.BCEWithLogitsLoss(pos_weight=keyword_pos_weight)
trait_loss = nn.BCEWithLogitsLoss(pos_weight=trait_pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
print("\n[3/5] Creating the neural network...")
print("  architecture: Embedding -> bidirectional GRU -> shared classifier")
print("  heads: keyword_head and trait_head (multi-label sigmoid outputs)")
print(f"  parameters={sum(parameter.numel() for parameter in model.parameters()):,}")
print("  optimizer: AdamW(lr=0.002, weight_decay=0.0001), loss: weighted BCE")


[3/5] Creating the neural network...
  architecture: Embedding -> bidirectional GRU -> shared classifier
  heads: keyword_head and trait_head (multi-label sigmoid outputs)
  parameters=1,479,325
  optimizer: AdamW(lr=0.002, weight_decay=0.0001), loss: weighted BCE


## Stage 5: training, evaluation, and standalone inference

The threshold is applied after sigmoid, while top-k limits traits and keywords separately. One `predict_spell` call obtains all labels directly from the same model.

In [5]:
def move_batch(batch: dict[str, Tensor]) -> dict[str, Tensor]:
    return {key: value.to(DEVICE) for key, value in batch.items()}

def run_epoch(loader: DataLoader[dict[str, Tensor]], training: bool) -> float:
    model.train(training)
    total_loss = 0.0
    for raw_batch in loader:
        batch = move_batch(raw_batch)
        with torch.set_grad_enabled(training):
            logits = model(batch["input_ids"], batch["rank"])
            loss = keyword_loss(logits["keywords"], batch["keyword_labels"])
            loss = loss + trait_loss(logits["traits"], batch["trait_labels"])
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        total_loss += loss.item() * batch["input_ids"].size(0)
    return total_loss / len(loader.dataset)

print("\n[4/5] Training and validating...")
print("  each epoch: forward pass -> weighted BCE loss -> backpropagation -> validation")
best_valid_loss = float("inf")
for epoch in range(1, 16):
    train_loss = run_epoch(train_loader, training=True)
    valid_loss = run_epoch(valid_loader, training=False)
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), ARTIFACT_DIR / "model.pt")
        checkpoint_note = " * best checkpoint saved"
    else:
        checkpoint_note = ""
    print(f"  epoch={epoch:02d} train_loss={train_loss:.4f} valid_loss={valid_loss:.4f}{checkpoint_note}")

model.load_state_dict(torch.load(ARTIFACT_DIR / "model.pt", map_location=DEVICE, weights_only=True))
print(f"  restored best checkpoint: {ARTIFACT_DIR / 'model.pt'} (valid_loss={best_valid_loss:.4f})")
print("  running test inference and comparing predictions with held-out labels...")

@torch.no_grad()
def collect_predictions(loader: DataLoader[dict[str, Tensor]]) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    keyword_probabilities: list[np.ndarray] = []
    trait_probabilities: list[np.ndarray] = []
    keyword_targets: list[np.ndarray] = []
    trait_targets: list[np.ndarray] = []
    for raw_batch in loader:
        batch = move_batch(raw_batch)
        outputs = model(batch["input_ids"], batch["rank"])
        keyword_probabilities.append(outputs["keywords"].sigmoid().cpu().numpy())
        trait_probabilities.append(outputs["traits"].sigmoid().cpu().numpy())
        keyword_targets.append(batch["keyword_labels"].cpu().numpy())
        trait_targets.append(batch["trait_labels"].cpu().numpy())
    return (np.concatenate(keyword_probabilities), np.concatenate(trait_probabilities),
            np.concatenate(keyword_targets), np.concatenate(trait_targets))

def multilabel_metrics(probabilities: np.ndarray, targets: np.ndarray, threshold: float = 0.35) -> dict[str, float]:
    predicted = probabilities >= threshold
    tp = np.logical_and(predicted, targets == 1).sum()
    fp = np.logical_and(predicted, targets == 0).sum()
    fn = np.logical_and(~predicted, targets == 1).sum()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    micro_f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    per_label_f1: list[float] = []
    for column in range(targets.shape[1]):
        column_tp = np.logical_and(predicted[:, column], targets[:, column] == 1).sum()
        column_fp = np.logical_and(predicted[:, column], targets[:, column] == 0).sum()
        column_fn = np.logical_and(~predicted[:, column], targets[:, column] == 1).sum()
        column_precision = column_tp / max(column_tp + column_fp, 1)
        column_recall = column_tp / max(column_tp + column_fn, 1)
        per_label_f1.append(2 * column_precision * column_recall / max(column_precision + column_recall, 1e-8))
    return {"micro_f1": float(micro_f1), "macro_f1": float(np.mean(per_label_f1)),
            "subset_accuracy": float(np.all(predicted == targets, axis=1).mean())}

test_keyword_probabilities, test_trait_probabilities, test_keyword_targets, test_trait_targets = collect_predictions(test_loader)
metrics = {"keywords": multilabel_metrics(test_keyword_probabilities, test_keyword_targets),
           "traits": multilabel_metrics(test_trait_probabilities, test_trait_targets)}
print("\n[5/5] Test results")
for group, values in metrics.items():
    print(f"  {group}: micro_f1={values["micro_f1"]:.3f}, macro_f1={values["macro_f1"]:.3f}, exact_match={values["subset_accuracy"]:.3f}")

@torch.no_grad()
def predict_probabilities(text: str, rank: int) -> dict[str, Tensor]:
    tokens = tokenize(text)[:MAX_TEXT_LENGTH]
    ids = [token_to_id.get(token, token_to_id["<UNK>"]) for token in tokens]
    ids += [token_to_id["<PAD>"]] * (MAX_TEXT_LENGTH - len(ids))
    input_ids = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    rank_tensor = torch.tensor([[rank / 10.0]], dtype=torch.float32, device=DEVICE)
    outputs = model(input_ids, rank_tensor)
    return {group: logits.sigmoid().squeeze(0).cpu() for group, logits in outputs.items()}

def decode_group(probabilities: Tensor, labels: list[str], threshold: float, top_k: int) -> list[dict[str, float | str]]:
    candidates = [(label, float(probability)) for label, probability in zip(labels, probabilities)
                  if probability >= threshold]
    candidates.sort(key=lambda item: item[1], reverse=True)
    return [{"label": label, "probability": round(probability, 4)} for label, probability in candidates[:max(0, top_k)]]

def predict_spell(text: str, rank: int, top_k_keywords: int = DEFAULT_TOP_K_KEYWORDS,
                  top_k_traits: int = DEFAULT_TOP_K_TRAITS, threshold: float = 0.35) -> dict[str, Any]:
    probabilities = predict_probabilities(text, rank)
    return {
        "text": text,
        "rank": rank,
        "keywords": decode_group(probabilities["keywords"], keyword_labels, threshold, top_k_keywords),
        "traits": decode_group(probabilities["traits"], trait_labels, threshold, top_k_traits),
        "probabilities": {"keywords": {label: round(float(probabilities["keywords"][index]), 6) for index, label in enumerate(keyword_labels)},
                          "traits": {label: round(float(probabilities["traits"][index]), 6) for index, label in enumerate(trait_labels)}},
        "model_version": "shared-encoder-two-heads-v1",
    }

metadata = {"token_to_id": token_to_id, "keyword_labels": keyword_labels, "trait_labels": trait_labels, "max_text_length": MAX_TEXT_LENGTH,
            "default_top_k_keywords": DEFAULT_TOP_K_KEYWORDS, "default_top_k_traits": DEFAULT_TOP_K_TRAITS,
            "seed": SEED, "model_version": "shared-encoder-two-heads-v1"}
(ARTIFACT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"  saved: {ARTIFACT_DIR / 'metadata.json'}, {ARTIFACT_DIR / 'metrics.json'}, {ARTIFACT_DIR / 'model.pt'}")
example = predict_spell("A burst of molten fire fills a 30-foot area. Creatures attempt a Reflex save and take persistent fire damage.", rank=3)
print("\nExample inference")
print(f"  input rank={example["rank"]}, text tokens={len(tokenize(example["text"]))}")
print("  keywords:", [(item["label"], item["probability"]) for item in example["keywords"]])
print("  traits:", [(item["label"], item["probability"]) for item in example["traits"]])
# This JSON is produced entirely by one model.
# No second model or co-occurrence post-processing is used.



[4/5] Training and validating...
  each epoch: forward pass -> weighted BCE loss -> backpropagation -> validation


  epoch=01 train_loss=1.1738 valid_loss=1.0243 * best checkpoint saved


  epoch=02 train_loss=0.9980 valid_loss=0.9600 * best checkpoint saved


  epoch=03 train_loss=0.9248 valid_loss=0.9372 * best checkpoint saved


  epoch=04 train_loss=0.8477 valid_loss=0.8982 * best checkpoint saved


...

  epoch=12 train_loss=0.4678 valid_loss=1.0679


  epoch=13 train_loss=0.4386 valid_loss=1.1367


  epoch=14 train_loss=0.4210 valid_loss=1.1201


  epoch=15 train_loss=0.4037 valid_loss=1.1549
  restored best checkpoint: /Users/bohdanlaskavyi/Fontys/year2september3/Personall challenge/PF2e-splells-creations/iterations/iteration_1/artifacts/model.pt (valid_loss=0.8736)
  running test inference and comparing predictions with held-out labels...



[5/5] Test results
  keywords: micro_f1=0.146, macro_f1=0.029, exact_match=0.000
  traits: micro_f1=0.388, macro_f1=0.156, exact_match=0.000
  saved: /Users/bohdanlaskavyi/Fontys/year2september3/Personall challenge/PF2e-splells-creations/iterations/iteration_1/artifacts/metadata.json, /Users/bohdanlaskavyi/Fontys/year2september3/Personall challenge/PF2e-splells-creations/iterations/iteration_1/artifacts/metrics.json, /Users/bohdanlaskavyi/Fontys/year2september3/Personall challenge/PF2e-splells-creations/iterations/iteration_1/artifacts/model.pt

Example inference
  input rank=3, text tokens=14
  keywords: [('reflex', 0.9818), ('damage', 0.9637), ('save', 0.958), ('fire', 0.9256), ('unaffected', 0.9191)]
  traits: [('fire', 0.9677), ('air', 0.9581), ('primal', 0.942), ('manipulate', 0.9218), ('arcane', 0.9124)]


## Test: the original idea – only rank + description

The original idea of the project: **the user types only the rank and the description of a spell, and the model fills in the rest.**

Iteration 1 already uses exactly this input: **rank + description**. So the test result is the result above, on the same 192 test spells as every other iteration. The model predicts only traits + traditions (the 100 labels); save, area, damage types and sustained were added in later iterations.

In [6]:
pd.DataFrame({"iteration 1, rank + description": {
    "traits + traditions, 100 labels (micro-F1)": round(metrics["traits"]["micro_f1"], 3),
    "save, area, damage types, sustained": "not predicted"}})

,"iteration 1, rank + description"
"traits + traditions, 100 labels (micro-F1)",0.388
"save, area, damage types, sustained",not predicted


### What the model makes of rank + description

Two homebrew spells and three spells from the test set (with the correct Foundry answer below them). The same examples are used in every iteration.

In [7]:
HOMEBREW = [
    ("Lava Geyser (homebrew, project proposal)", 4,
     "You channel magical energy to rupture the earth, summoning a burst of molten rock. The eruption deals massive heat damage "
     "to anyone in the area and requires you to perform complex hand gestures to sustain it. Each creature in the area must "
     "attempt a basic Reflex save."),
    ("Molten Burst (homebrew, also used in iteration 1)", 3,
     "A burst of molten fire fills a 30-foot area. Creatures attempt a Reflex save and take persistent fire damage."),
]
REAL_SPELLS = ["Gust of Wind", "Suggestion", "Wordsmith"]  # from the test set; Wordsmith is a focus spell

def predicted_labels(rank, description):
    # predict_spell from above: threshold 0.35, at most 5 labels.
    return ", ".join(item["label"] for item in predict_spell(description, rank)["traits"]) or "–"

rows = [{"spell": name, "rank": rank, "traits + traditions": predicted_labels(rank, description)} for name, rank, description in HOMEBREW]
for name in REAL_SPELLS:
    spell = test_df[test_df["name"] == name].iloc[0]
    rows.append({"spell": f"{name} (test set)", "rank": spell["rank"], "traits + traditions": predicted_labels(spell["rank"], spell["text"])})
    rows.append({"spell": "↳ correct answer", "rank": spell["rank"], "traits + traditions": ", ".join(spell["traits"])})
pd.DataFrame(rows).set_index("spell")

,rank,traits + traditions
spell,,
"Lava Geyser (homebrew, project proposal)",4,"primal, manipulate, air, fire, earth"
"Molten Burst (homebrew, also used in iteration 1)",3,"fire, air, primal, manipulate, arcane"
Gust of Wind (test set),1,"air, primal, arcane, manipulate, fire"
↳ correct answer,1,"air, arcane, concentrate, manipulate, primal"
Suggestion (test set),4,"mental, incapacitation, illusion, occult, visual"
↳ correct answer,4,"arcane, concentrate, incapacitation, linguisti..."
Wordsmith (test set),4,"cleric, focus, occult, fortune, cantrip"
↳ correct answer,4,"cleric, concentrate, focus, manipulate"


### Test result

**0.388**: lower than always answering `concentrate, manipulate` (0.473).

In the examples it predicts `fire` and `air` for three of the five spells (also for Gust of Wind, which has no fire) and never predicts `concentrate`, which almost every spell has. The reasons (wrong targets, too little data to learn word vectors from scratch, Foundry links removed from the text) are fixed in iteration 1.5.